In [1]:
%%capture
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.preprocessing import LabelEncoder
from recsys_pipeliner.recommendations.transformer import (
    SimilarityTransformer,
    UserItemMatrixTransformer,
)
from recsys_pipeliner.algorithms.recommenders import ItemBasedCFRecommender
from IPython.display import display
from recsys_pipeliner.dataset import EvaluationDataset

In [3]:
# load test data
data_types = {"user_id": str, "item_id": str, "rating": np.float64}
user_item_ratings = pd.read_csv(
    "../../tests/test_data/user_item_ratings_toy.csv", dtype=data_types
)

# encode the user/item ids
item_encoder = LabelEncoder()
user_encoder = LabelEncoder()

user_item_ratings["item_id"] = item_encoder.fit_transform(user_item_ratings["item_id"])
user_item_ratings["user_id"] = user_encoder.fit_transform(user_item_ratings["user_id"])

unique_users = pd.Series(user_encoder.classes_)
unique_items = pd.Series(item_encoder.classes_)

# create the user/item matrix
user_item_matrix_transformer = UserItemMatrixTransformer()

user_item_matrix = user_item_matrix_transformer.transform(
    user_item_ratings.to_numpy(),
)

# sanity check
users = user_item_ratings["user_id"].to_numpy().astype(int)
items = user_item_ratings["item_id"].to_numpy().astype(int)
ratings = user_item_ratings["rating"].to_numpy().astype(np.float32)
for user, item, rating in zip(users, items, ratings):
    assert user_item_matrix[user, item] == rating

In [4]:
dataset = EvaluationDataset(user_item_ratings)
loo_iterator = dataset.leave_one_out()

print("user_item_ratings", user_item_ratings.shape)
for trainset, testset in loo_iterator:
    print("trainset", trainset.shape)
    print("testset", testset.shape)


user_item_ratings (96, 3)
trainset (84, 3)
testset (12, 3)


In [5]:
anti_testset = dataset.anti_testset

users = anti_testset[:, 0]
items = anti_testset[:, 1]

# sanity check
for user, item in anti_testset:
    assert user_item_matrix[user, item] == 0